## Quickstart
https://docs.langchain.com/oss/python/langchain/quickstart


In [2]:
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage, AIMessage, SystemMessage

In [3]:

model = ChatOpenAI(
    model="openai/gpt-4o-mini",   # or any OpenRouter-supported model
    base_url="https://models.github.ai/inference",
    api_key='ghp_xxxxxxx',
    max_tokens=400,
    temperature=0.9
)

In [ ]:
response = model.invoke([SystemMessage(content="You are a helpful assistant."), HumanMessage(content="What is the capital of West Bengal?")])
print(response.content)


The capital of West Bengal is Kolkata.
content='The capital of West Bengal is Kolkata.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 25, 'total_tokens': 34, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 9, 'engine_ttft_ms': 38, 'engine_ttlt_ms': 115, 'pre_inference_ms': 96, 'service_tbt_ms': 9, 'service_ttft_ms': 510, 'service_ttlt_ms': 584, 'total_duration_ms': 495, 'user_visible_ttft_ms': 414}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_eb37e061ec', 'id': 'chatcmpl-DePjEcC0tJJbhpyKX5IyHM21Svw1G', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019e1850-8887-7602-b5c1-d4bcf3f5a142-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'inp

### Build a basic agent

In [5]:
from langchain.agents import create_agent

def get_weather(city: str) -> str:
    """Get weather for a given city."""
    
    # In a real implementation, this would call a weather API.
    return f"The current weather in {city} is sunny with a temperature of 25°C."

agent = create_agent(
    model=model, 
    tools=[get_weather],
    system_prompt="You are a helpful assistant",)

result = agent.invoke({"messages": [HumanMessage(content="What is the weather in New York?")]})
print(result)
print(result["messages"][-1].content)

{'messages': [HumanMessage(content='What is the weather in New York?', additional_kwargs={}, response_metadata={}, id='68d76246-10bf-463d-9e31-52104f501db5'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 58, 'total_tokens': 74, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 8, 'engine_ttft_ms': 31, 'engine_ttlt_ms': 154, 'pre_inference_ms': 148, 'service_tbt_ms': 11, 'service_ttft_ms': 998, 'service_ttlt_ms': 1175, 'total_duration_ms': 1027, 'user_visible_ttft_ms': 851}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_eb37e061ec', 'id': 'chatcmpl-DePOc5e1YXfJ89kLSXW1A5lR2UQT7', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id

### Build a real-world agent

In the following example we will build a research agent that can answer questions about text files. Along the way we will explore the following concepts:
1. Detailed system prompts for better agent behavior
2. Create tools that integrate with external data
3. Model configuration for consistent responses
4. Conversational memory for chat-like interactions
5. Deep Agents for built-in features
6. Testing your agent

In [6]:
#Define the system prompt

SYSTEM_PROMPT = """You are a literary data assistant.

## Capabilities

- `fetch_text_from_url`: loads document text from a URL into the conversation.
Do not guess line counts or positions—ground them in tool results from the saved file."""

In [13]:
#Create tools
#This example uses a tool to load a document from a given URL:

from urllib.error import URLError
from urllib.request import Request, urlopen

from langchain.tools import tool
from sqlalchemy import text


@tool
def fetch_text_from_url(url: str) -> str:
    """Fetch the document from a URL.
    """
    req = Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 (compatible; quickstart-research/1.0)"},
    )
    try:
        with urlopen(req, timeout=120) as resp:
            raw = resp.read()
    except URLError as e:
        return f"Fetch failed: {e}"
    text = raw.decode("utf-8", errors="replace")
    return text[:1000]
    #return text

In [ ]:
#Configure model

# we did this above, but we can also do it here when we create the agent:

In [14]:
#Add memory
#Add memory to the agent to maintain state across interactions. This allows the agent to remember previous conversations and context.
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

In [15]:
# Create and run the agent
# Now assemble your agent with all the components and run it.
# There are two different frameworks for creating agents: LangChain agents and deep agents. Both LangChain and deep agents provide you with fine-grained control over tools, memory, and more. 
# The main difference between both is that deep agents come with a range of commonly useful capabilities already built in, such as planning, file system tools, and subagents.
# Use deep agents when you want maximum capability with minimal setup; choose LangChain agents when you need fine-grained control.


from langchain.agents import create_agent
#from deepagents import create_deep_agent

agent = create_agent(
    model=model,
    tools=[fetch_text_from_url],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
)

#deep_agent = create_deep_agent(
#    model=model,
#    tools=[fetch_text_from_url],
#    system_prompt=SYSTEM_PROMPT,
#    checkpointer=checkpointer,
#)

content = f"""Project Gutenberg hosts a full plain-text copy of F. Scott Fitzgerald's The Great Gatsby.
URL: https://www.gutenberg.org/files/64317/64317-0.txt

Answer as much as you can:

1) How many lines in the complete Gutenberg file contain the substring `Gatsby` (count lines, not occurrences within a line, each line ends with a line break).
2) The 1-based line number of the first line in the file that contains `Daisy`.
3) A two-sentence neutral synopsis.

Do your best on (1) and (2). If at any point you realize you cannot **verify** an exact answer with
your available tools and reasoning, do not fabricate numbers: use `null` for that field and spell out
the limitation in `how_you_computed_counts`. If you encounter any errors please report what the error was and what the error message was."""

agent_result = agent.invoke(
    {"messages": [{"role": "user", "content": content}]},
    config={"configurable": {"thread_id": "great-gatsby-lc"}},
)
#deep_agent_result = deep_agent.invoke(
#    {"messages": [{"role": "user", "content": content}]},
#    config={"configurable": {"thread_id": "great-gatsby-da"}},
#)
print(agent_result["messages"][-1].content_blocks)
print("\n")
#print(deep_agent_result["messages"][-1].content_blocks)

[{'type': 'text', 'text': 'It seems that the fetching of the text is consistently interrupted and does not yield the entire document. This prevents me from performing a complete analysis.\n\nAs a result, I must return null for my findings on the line counts and positions, as I cannot verify the data accurately. Here’s the summary of the findings:\n\n1) Lines containing the substring `Gatsby`: `null` (unable to count lines without complete text).\n2) First line containing `Daisy`: `null` (unable to locate the line without complete text).\n3) Synopsis: "The Great Gatsby explores themes of wealth, love, and the American Dream through the life of Jay Gatsby and his obsession with Daisy Buchanan. Set in the 1920s, the story highlights the moral decay and social stratification of the era."\n\n### Limitations\nThe limitation stems from the fact that I could not retrieve the entire text of "The Great Gatsby" from the provided URL due to repeated fetch failures, leading to incompleteness in bot